# Extracción de Datos
Tenemos diferentes conjuntos de datos, para los datos satélitales tenemos 2 archivos csv, para la temperatura, precipitación, elevación y pendiente tenemos diferentes tipos de rasters, y tenemos shape files que nos dan la geometría de la parcela. Para la temperatura y elevación tenemos rasters para cada mes desde del 2022 a 2025. En caso de la elevación y pendiente es solo un raster.

El objetivo es de extraer los datos para crear un solo archivo csv.

In [2]:
import pandas as pd 
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.plot import plotting_extent
from rasterio.windows import from_bounds
import geopandas as gpd
import matplotlib.pyplot as plt
from pyproj import Transformer
import shapely
from rasterio.mask import mask
from shapely.geometry import mapping
from shapely.geometry import box
from rasterio.transform import xy
from collections import Counter
import scipy
from pathlib import Path

In [3]:
parcelas_gdf = gpd.read_file('../../DataSet/DATASET_RETO_AGRO_2026/Parcelas_Reto_AGC_CONJUNTO_70_30/Parcelas_Reto_AGC_CONJUNTO.shp')

parcelas_gdf.head()

,Cultivo,ID_POLIGON,área_ha,Municipio,Estado,CONJUNTO,geometry
0,Cebada,AGC_196,8.57,Emiliano Zapata,Hidalgo,ENTRENAMIENTO,"POLYGON Z ((-98.57739 19.65802 0, -98.57751 19..."
1,Cebada,AGC_044,0.68,Emiliano Zapata,Hidalgo,ENTRENAMIENTO,"POLYGON Z ((-98.58655 19.62943 0, -98.5862 19...."
2,Cebada,AGC_020,8.34,Emiliano Zapata,Hidalgo,ENTRENAMIENTO,"POLYGON Z ((-98.58494 19.6278 0, -98.58656 19...."
3,CEBADA,AGC_174,6.89,Cuautepec de Hinojosa,Hidalgo,PREDICCION,"POLYGON Z ((-98.30604 19.9 0, -98.30486 19.899..."
4,CEBADA,AGC_083,2.87,Cuautepec de Hinojosa,Hidalgo,ENTRENAMIENTO,"POLYGON Z ((-98.29383 19.8882 0, -98.29378 19...."


In [4]:
print(f'Shape: {parcelas_gdf.shape}')
print(f'CRS: {parcelas_gdf.crs}')
print(f'Columns: {parcelas_gdf.columns}')

Shape: (197, 7)
CRS: EPSG:4326
Columns: Index(['Cultivo', 'ID_POLIGON', 'área_ha', 'Municipio', 'Estado', 'CONJUNTO',
       'geometry'],
      dtype='str')


## Datos Topográficos.

### Raster de Elevación.

In [5]:
elev_path = '../../DataSet/DATASET_RETO_AGRO_2026/Reto_AgroCebada_Topografia_INEGI_CEM4/Reto_AgroCebada_Topografia_INEGI_CEM4/Reto_AgroCebada_Topografia_INEGI_CEM4/Elevacion_INEGI_CEM4_120m.tif'

print('Elevación:')

with rasterio.open(elev_path) as src_elev:
    print(f'CSR: {src_elev.crs}')
    print(f'Dimensiones: {src_elev.width} x {src_elev.height}')
    print(f'Bandas: {src_elev.count}')
    print(f'Resolución: {src_elev.res}')
    print(f'Bounds: {src_elev.bounds}')
    print(f'No Data: {src_elev.nodata}')
    print(f'Data type: {src_elev.dtypes[0]}')

Elevación:
CSR: EPSG:6372
Dimensiones: 2900 x 3362
Bandas: 1
Resolución: (120.0, 120.0)
Bounds: BoundingBox(left=2715840.0, bottom=650400.0, right=3063840.0, top=1053840.0)
No Data: -9999.0
Data type: float32


In [6]:
#Función para obtner un dataframe que relacione la parcela con la elevación

def get_elevation(parcela, elev_path):
    with rasterio.open(elev_path) as src:
        
        parcela_raster = parcela.to_crs(src.crs) #Pasamos la parcela a mismo formato del raster
        
        #Obtenemos el poligono de la parcela
        geom_parcela = parcela_raster.geometry.iloc[0]
        geom = [mapping(geom_parcela)]
        
        #Recortamos la parcela alrededor del raster
        parcela_values, transform = mask(src, geom, crop=True, filled=False, all_touched=True)
        
        #Obtenemos el area de la parcela y obtenemos el area que tiene cada pixel dentro de la parcel
        area_total = geom_parcela.area
        area_valida = 0
        average_sum = 0
        
        for i in range(parcela_values.shape[1]):
            for j in range(parcela_values.shape[2]):
                
                if parcela_values.mask[0, i, j]:
                    continue #Ignoramos los No Data
                    
                #Obtemos el valor del raster
                elevacion = parcela_values[0, i, j]
                
                #Valores del pixel
                x_left, y_up = xy(transform, i, j, offset="ul")
                x_right, y_down = xy(transform, i, j, offset="lr")
                
                # Geometría del píxel
                pixel = box(x_left, y_down, x_right, y_up)
                
                #Obtenemos la parte del pixel que pertenece a la parcela
                intersection = pixel.intersection(geom_parcela)
                area_intersection = intersection.area
                area_valida += area_intersection
                
                average_sum += (float(elevacion) * area_intersection)
        
        # Media ponderada
        if area_valida > 0:
            elevacion_media = average_sum / area_valida
        else:
            elevacion_media = np.nan
        
        # Cobertura válida
        cobertura = area_valida / area_total
        
        return {
            "area_total_m2": area_total,
            "area_valida_m2": area_valida,
            "cobertura_elevacion": cobertura,
            "cobertura_elevacion_pct": cobertura * 100,
            "elevacion_media": elevacion_media
        } 

In [7]:
#Para las 197 parcelas

resultados_elevacion = []

for _, parcela in parcelas_gdf.iterrows():

    resultado = get_elevation(
        gpd.GeoDataFrame(
            [parcela],
            crs=parcelas_gdf.crs
        ),
        elev_path
    )
    
    resultado["ID_POLIGON"] = parcela["ID_POLIGON"]
    
    resultados_elevacion.append(resultado)

In [8]:
df_elevacion = pd.DataFrame(resultados_elevacion)

df_elevacion = df_elevacion[
    [
        "ID_POLIGON",
        "area_total_m2",
        "area_valida_m2",
        "cobertura_elevacion",
        "cobertura_elevacion_pct",
        "elevacion_media"
    ]
]

df_elevacion.head()

,ID_POLIGON,area_total_m2,area_valida_m2,cobertura_elevacion,cobertura_elevacion_pct,elevacion_media
0,AGC_196,85130.668788,85130.668788,1.000000,100.000000,2543.659752
1,AGC_044,6760.402971,6760.402971,1.000000,100.000000,2529.993625
2,AGC_020,82856.226598,81059.998963,0.978321,97.832115,2528.924828
3,AGC_174,68423.862739,68423.862739,1.000000,100.000000,2546.616547
4,AGC_083,28466.739798,28466.739798,1.000000,100.000000,2564.037732


In [9]:
df_elevacion["elevacion_media"].describe()

count     197.000000
mean     2667.102482
std        92.283855
min      2496.648468
25%      2611.351962
50%      2638.902931
75%      2726.683106
max      2906.039520
Name: elevacion_media, dtype: float64

In [10]:
print("Número de parcelas:", len(df_elevacion))
print("Elevaciones NaN:", df_elevacion["elevacion_media"].isna().sum())
print("Coberturas NaN:", df_elevacion["cobertura_elevacion_pct"].isna().sum())

Número de parcelas: 197
Elevaciones NaN: 0
Coberturas NaN: 0


In [11]:
print(
    df_elevacion[
        df_elevacion["cobertura_elevacion_pct"] < 99.999
    ][
        ["ID_POLIGON", "cobertura_elevacion_pct", "elevacion_media"]
    ]
)


    ID_POLIGON  cobertura_elevacion_pct  elevacion_media
2      AGC_020                97.832115      2528.924828
163    AGC_023                95.899497      2502.457298
175    AGC_170                75.938015      2501.227723
178    AGC_164                95.061206      2496.648468


### Raster Pendiente

In [12]:
pend_path = '../../DataSet/DATASET_RETO_AGRO_2026/Reto_AgroCebada_Topografia_INEGI_CEM4/Reto_AgroCebada_Topografia_INEGI_CEM4/Reto_AgroCebada_Topografia_INEGI_CEM4/Pendiente_INEGI_CEM4_120m_grados.tif'

In [13]:
def get_slope(parcela, raster_path):
    
    with rasterio.open(raster_path) as src:
        
        # Reproyectar parcela al CRS del raster
        parcela_raster = parcela.to_crs(src.crs)
        
        # Geometría de la parcela
        geom_parcela = parcela_raster.geometry.iloc[0]
        geom = [mapping(geom_parcela)]
        
        # Recortar raster alrededor de la parcela
        valores, transform = mask(
            src,
            geom,
            crop=True,
            filled=False,
            all_touched=True
        )
        
        # Área total
        area_total = geom_parcela.area
        
        area_valida = 0
        suma_ponderada = 0
        
        # Recorrer píxeles
        for fila in range(valores.shape[1]):
            for columna in range(valores.shape[2]):
                
                # Ignorar NoData
                if valores.mask[0, fila, columna]:
                    continue
                
                # Valor de pendiente
                pendiente = valores[0, fila, columna]
                
                # Coordenadas del píxel
                x_izq, y_sup = rasterio.transform.xy(
                    transform,
                    fila,
                    columna,
                    offset="ul"
                )
                
                x_der, y_inf = rasterio.transform.xy(
                    transform,
                    fila,
                    columna,
                    offset="lr"
                )
                
                # Geometría del píxel
                pixel = box(
                    x_izq,
                    y_inf,
                    x_der,
                    y_sup
                )
                
                # Intersección con la parcela
                interseccion = pixel.intersection(geom_parcela)
                
                area_interseccion = interseccion.area
                
                # Área válida
                area_valida += area_interseccion
                
                # Pendiente × área
                suma_ponderada += (
                    float(pendiente) *
                    area_interseccion
                )
        
        # Media ponderada
        if area_valida > 0:
            pendiente_media = suma_ponderada / area_valida
        else:
            pendiente_media = np.nan
        
        # Cobertura
        cobertura = area_valida / area_total
        
        return {
            "area_total_m2": area_total,
            "area_valida_m2": area_valida,
            "cobertura_pendiente": cobertura,
            "cobertura_pendiente_pct": cobertura * 100,
            "pendiente_media": pendiente_media
        }

In [14]:
parcela_001 = parcelas_gdf[
    parcelas_gdf["ID_POLIGON"] == "AGC_001"
].copy()

resultado_001_pendiente = get_slope(
    parcela_001,
    pend_path
)

print(resultado_001_pendiente)

{'area_total_m2': 75685.78283383907, 'area_valida_m2': 75685.78283386417, 'cobertura_pendiente': 1.0000000000003317, 'cobertura_pendiente_pct': 100.00000000003317, 'pendiente_media': 3.039547531881103}


In [15]:
resultados_pendiente = []

for _, parcela in parcelas_gdf.iterrows():

    resultado = get_slope(gpd.GeoDataFrame([parcela], crs=parcelas_gdf.crs),pend_path)
    
    resultado["ID_POLIGON"] = parcela["ID_POLIGON"]
    
    resultados_pendiente.append(resultado)

In [16]:
df_pendiente = pd.DataFrame(resultados_pendiente)

df_pendiente = df_pendiente[
    [
        "ID_POLIGON",
        "area_total_m2",
        "area_valida_m2",
        "cobertura_pendiente",
        "cobertura_pendiente_pct",
        "pendiente_media"
    ]
]

In [17]:
print(df_pendiente.head())

  ID_POLIGON  area_total_m2  area_valida_m2  cobertura_pendiente  \
0    AGC_196   85130.668788    85130.668788             1.000000   
1    AGC_044    6760.402971     6626.924578             0.980256   
2    AGC_020   82856.226598    54063.973641             0.652503   
3    AGC_174   68423.862739    68423.862739             1.000000   
4    AGC_083   28466.739798    28466.739798             1.000000   

   cobertura_pendiente_pct  pendiente_media  
0               100.000000         1.740829  
1                98.025585         0.348862  
2                65.250345         0.561396  
3               100.000000         2.284557  
4               100.000000         2.368227  


### Elevación + Pendiente

In [18]:
df_topografia = df_elevacion.merge(df_pendiente, on='ID_POLIGON', how='inner', suffixes=('_elevacion', '_pendiente'))

df_topografia = df_topografia[
    [
        "ID_POLIGON",
        "elevacion_media",
        "cobertura_elevacion_pct",
        "pendiente_media",
        "cobertura_pendiente_pct"
    ]
]

In [19]:
df_topografia.head()

,ID_POLIGON,elevacion_media,cobertura_elevacion_pct,pendiente_media,cobertura_pendiente_pct
0,AGC_196,2543.659752,100.000000,1.740829,100.000000
1,AGC_044,2529.993625,100.000000,0.348862,98.025585
2,AGC_020,2528.924828,97.832115,0.561396,65.250345
3,AGC_174,2546.616547,100.000000,2.284557,100.000000
4,AGC_083,2564.037732,100.000000,2.368227,100.000000


## Datos climatológicos

### Rasters Precipitación.

In [20]:
precipitaion_dir = Path('../../DataSet/DATASET_RETO_AGRO_2026/Reto_AgroCebada_CHIRPS_Precipitacion_2022_2025/Reto_AgroCebada_CHIRPS_Precipitacion_2022_2025/Precipitacion_mensual/')
precipitation_files = sorted(precipitaion_dir.glob("*/*.tif"))

print(f'Número total de archivos: {len(precipitation_files)}')

for archivo in precipitation_files[:5]:
    print(archivo.name)


Número total de archivos: 48
PREC_2022_01.tif
PREC_2022_02.tif
PREC_2022_03.tif
PREC_2022_04.tif
PREC_2022_05.tif


In [21]:
def get_file_date(root):
    file_no_extension = root.with_suffix('') #To delete extendion .tif
    file_name = file_no_extension.name
    file_name_elements = file_name.split('_')
    variable_name = file_name_elements[0]
    year = int(file_name_elements[1])
    month = int(file_name_elements[2])
        
    return variable_name, year, month

In [22]:
def get_data_from_raster(raster_path, parcela, variable_name):
    with rasterio.open(raster_path) as src:
        parcela_geom = parcela.to_crs(src.crs)
        parcela_geom = parcela_geom.geometry.iloc[0]
        
        #Recortamos los pixeles del raster que intersacatn con la parcela
        values, transform = mask(src, [mapping(parcela_geom)], crop=True, filled=False, all_touched=True)
        
        #Objeto para transformar de coordendas a metros
        transfomer = Transformer.from_crs(src.crs, 'EPSG:6372', always_xy=True)
        
        #Parcela en metros
        parcela_m = parcela.to_crs('EPSG:6372')
        geometry_m = parcela_m.geometry.iloc[0]
        
        area_total = geometry_m.area
        area_valida = 0
        suma_ponderada = 0
        pixeles_validos = 0
        
        for i in range(values.shape[1]):
            for j in range(values.shape[2]):
                
                #Ignoramos los pixeles que no tienen datos
                if  np.ma.getmaskarray(values)[0, i, j]:
                    continue
                
                valor = float(values[0, i, j])

                if not np.isfinite(valor):
                    continue

                # Límites del píxel en el CRS del raster
                x1, y1 = xy(transform, i, j, offset="ul")

                x2, y2 = xy(transform, i, j, offset="lr")

                pixel_geo = box(x1, y2, x2, y1)

                # Transformar el píxel a coordenadas métricas
                pixel_m = shapely.ops.transform(transfomer.transform,pixel_geo)

                # Intersección real con la parcela
                interseccion = pixel_m.intersection(geometry_m)
                area_interseccion = interseccion.area

                if area_interseccion <= 0:
                    continue

                area_valida += area_interseccion
                suma_ponderada += valor * area_interseccion
                pixeles_validos += 1

        media_ponderada = (
            suma_ponderada / area_valida
            if area_valida > 0
            else np.nan
        )

        return {
            "area_total_m2": area_total,
            "area_valida_m2": area_valida,
            "cobertura_pct": 100 * area_valida / area_total,
            f"{variable_name}_media": media_ponderada,
            "pixeles_validos": pixeles_validos
        }

In [23]:
for file in precipitation_files[:5]:
    variable, year, month = get_file_date(file)
    print(f'{file.name} ---> year: {year}, month: {month}')

PREC_2022_01.tif ---> year: 2022, month: 1
PREC_2022_02.tif ---> year: 2022, month: 2
PREC_2022_03.tif ---> year: 2022, month: 3
PREC_2022_04.tif ---> year: 2022, month: 4
PREC_2022_05.tif ---> year: 2022, month: 5


In [24]:
def process_monthly_precipitation(root, parcelas_gdf):
    
    variable_name, year, month = get_file_date(root)
    results = []

    for _, parcela in parcelas_gdf.iterrows():

        parcela_gdf = gpd.GeoDataFrame(
            [parcela],
            geometry="geometry",
            crs=parcelas_gdf.crs
        )

        result = get_data_from_raster(str(root), parcela_gdf, 'precipitacion')

        results.append({
            "ID_POLIGON": parcela["ID_POLIGON"],
            "año": year,
            "mes": month,
            "precipitacion_mm": result["precipitacion_media"],
            "cobertura_pct": result["cobertura_pct"],
            "pixeles_validos": result["pixeles_validos"]
        })

    return pd.DataFrame(results)

In [25]:
results_precipitation_all = []

for i, root in enumerate(precipitation_files, start=1):
    df_mes = process_monthly_precipitation(root, parcelas_gdf)
    results_precipitation_all.append(df_mes)
    
    print(f'{i}/{len(precipitation_files)}: {root.name}')

#unimos todos los meses
df_precipitation = pd.concat(results_precipitation_all, ignore_index=True)

print("\nDimensiones:", df_precipitation.shape)
print("Parcelas únicas:", df_precipitation["ID_POLIGON"].nunique())
print("Años:", sorted(df_precipitation["año"].unique()))
print("Meses:", sorted(df_precipitation["mes"].unique()))

1/48: PREC_2022_01.tif
2/48: PREC_2022_02.tif
3/48: PREC_2022_03.tif
4/48: PREC_2022_04.tif
5/48: PREC_2022_05.tif
6/48: PREC_2022_06.tif
7/48: PREC_2022_07.tif
8/48: PREC_2022_08.tif
9/48: PREC_2022_09.tif
10/48: PREC_2022_10.tif
11/48: PREC_2022_11.tif
12/48: PREC_2022_12.tif
13/48: PREC_2023_01.tif
14/48: PREC_2023_02.tif
15/48: PREC_2023_03.tif
16/48: PREC_2023_04.tif
17/48: PREC_2023_05.tif
18/48: PREC_2023_06.tif
19/48: PREC_2023_07.tif
20/48: PREC_2023_08.tif
21/48: PREC_2023_09.tif
22/48: PREC_2023_10.tif
23/48: PREC_2023_11.tif
24/48: PREC_2023_12.tif
25/48: PREC_2024_01.tif
26/48: PREC_2024_02.tif
27/48: PREC_2024_03.tif
28/48: PREC_2024_04.tif
29/48: PREC_2024_05.tif
30/48: PREC_2024_06.tif
31/48: PREC_2024_07.tif
32/48: PREC_2024_08.tif
33/48: PREC_2024_09.tif
34/48: PREC_2024_10.tif
35/48: PREC_2024_11.tif
36/48: PREC_2024_12.tif
37/48: PREC_2025_01.tif
38/48: PREC_2025_02.tif
39/48: PREC_2025_03.tif
40/48: PREC_2025_04.tif
41/48: PREC_2025_05.tif
42/48: PREC_2025_06.tif
4

In [26]:

print(
    "Duplicados:",
    df_precipitation.duplicated(
        subset=["ID_POLIGON", "año", "mes"]
    ).sum()
)

print(
    "\nFilas por año:",
    df_precipitation.groupby("año").size().to_dict()
)

print(
    "\nParcelas por mes:",
    df_precipitation.groupby(["año", "mes"])
    ["ID_POLIGON"].nunique().value_counts().to_dict()
)

print(
    "\nValores faltantes:\n",
    df_precipitation[
        ["precipitacion_mm", "cobertura_pct"]
    ].isna().sum()
)

print(
    "\nCobertura mínima por mes:\n",
    df_precipitation.groupby(["año", "mes"])
    ["cobertura_pct"].min()
)


Duplicados: 0

Filas por año: {2022: 2364, 2023: 2364, 2024: 2364, 2025: 2364}

Parcelas por mes: {197: 48}

Valores faltantes:
 precipitacion_mm    0
cobertura_pct       0
dtype: int64

Cobertura mínima por mes:
 año   mes
2022  1      100.0
      2      100.0
      3      100.0
      4      100.0
      5      100.0
      6      100.0
      7      100.0
      8      100.0
      9      100.0
      10     100.0
      11     100.0
      12     100.0
2023  1      100.0
      2      100.0
      3      100.0
      4      100.0
      5      100.0
      6      100.0
      7      100.0
      8      100.0
      9      100.0
      10     100.0
      11     100.0
      12     100.0
2024  1      100.0
      2      100.0
      3      100.0
      4      100.0
      5      100.0
      6      100.0
      7      100.0
      8      100.0
      9      100.0
      10     100.0
      11     100.0
      12     100.0
2025  1      100.0
      2      100.0
      3      100.0
      4      100.0
      5      100

In [27]:
display(df_precipitation.head())

,ID_POLIGON,año,mes,precipitacion_mm,cobertura_pct,pixeles_validos
0,AGC_196,2022,1,6.847781,100.0,1
1,AGC_044,2022,1,8.457767,100.0,1
2,AGC_020,2022,1,8.457767,100.0,1
3,AGC_174,2022,1,19.384502,100.0,1
4,AGC_083,2022,1,21.224924,100.0,1


### Rasters Temperatura

In [28]:
tmax_path = '../../DataSet/DATASET_RETO_AGRO_2026/Reto_AgroCebada_Temperatura_2022_2025_RECORTADA_CHIRPS/Reto_AgroCebada_Temperatura_2022_2025/Tmax/2025/Tmax_2025_01.tif'

print('Tmax:')

with rasterio.open(tmax_path) as src_tmax:
    print(f'CSR: {src_tmax.crs}')
    print(f'Dimensiones: {src_tmax.width} x {src_tmax.height}')
    print(f'Bandas: {src_tmax.count}')
    print(f'Resolución: {src_tmax.res}')
    print(f'Bounds: {src_tmax.bounds}')
    print(f'No Data: {src_tmax.nodata}')
    print(f'Data type: {src_tmax.dtypes[0]}')

Tmax:
CSR: EPSG:4326
Dimensiones: 65 x 72
Bandas: 1
Resolución: (0.05000000074505806, 0.05000000074505806)
Bounds: BoundingBox(left=-99.94999880716205, bottom=17.799999222159386, right=-96.69999875873327, top=21.399999275803566)
No Data: -9999.0
Data type: float32


In [29]:
tmin_path = '../../DataSet/DATASET_RETO_AGRO_2026/Reto_AgroCebada_Temperatura_2022_2025_RECORTADA_CHIRPS/Reto_AgroCebada_Temperatura_2022_2025/Tmin/2025/Tmin_2025_01.tif'

print('Tmin')

with rasterio.open(tmin_path) as src_tmin:
    print(f'CSR: {src_tmin.crs}')
    print(f'Dimensiones: {src_tmin.width} x {src_tmin.height}')
    print(f'Bandas: {src_tmin.count}')
    print(f'Resolución: {src_tmin.res}')
    print(f'Bounds: {src_tmin.bounds}')
    print(f'No Data: {src_tmin.nodata}')
    print(f'Data type: {src_tmin.dtypes[0]}')

Tmin
CSR: EPSG:4326
Dimensiones: 65 x 72
Bandas: 1
Resolución: (0.05000000074505806, 0.05000000074505806)
Bounds: BoundingBox(left=-99.94999880716205, bottom=17.799999222159386, right=-96.69999875873327, top=21.399999275803566)
No Data: -9999.0
Data type: float32


In [30]:
temp_dir = Path('../../DataSet/DATASET_RETO_AGRO_2026/Reto_AgroCebada_Temperatura_2022_2025_RECORTADA_CHIRPS/Reto_AgroCebada_Temperatura_2022_2025/')

temp_files = sorted(temp_dir.rglob('*.tif'))

print(f'Total de archivos tif: {len(temp_files)}')

for file in temp_files[:10]:
    print(file.name)

Total de archivos tif: 96
Tmax_2022_01.tif
Tmax_2022_02.tif
Tmax_2022_03.tif
Tmax_2022_04.tif
Tmax_2022_05.tif
Tmax_2022_06.tif
Tmax_2022_07.tif
Tmax_2022_08.tif
Tmax_2022_09.tif
Tmax_2022_10.tif


In [31]:
resultados_temperatura = []

for i, ruta in enumerate(temp_files, start=1):
    variable, año, mes = get_file_date(ruta)

    nombre_variable = (
        "temperatura_maxima"
        if variable == "Tmax"
        else "temperatura_minima"
    )

    for _, parcela in parcelas_gdf.iterrows():
        parcela_gdf = gpd.GeoDataFrame(
            [parcela],
            geometry="geometry",
            crs=parcelas_gdf.crs
        )

        resultado = get_data_from_raster(
            ruta,
            parcela_gdf,
            nombre_variable
        )

        resultados_temperatura.append({
            "ID_POLIGON": parcela["ID_POLIGON"],
            "año": año,
            "mes": mes,
            "variable": variable,
            "temperatura_media": resultado[
                f"{nombre_variable}_media"
            ],
            "cobertura_pct": resultado["cobertura_pct"],
            "pixeles_validos": resultado["pixeles_validos"]
        })

    print(f"{i}/{len(temp_files)}: {ruta.name}")

df_temperatura = pd.DataFrame(resultados_temperatura)

print("\nDimensiones:", df_temperatura.shape)
print("Parcelas únicas:", df_temperatura["ID_POLIGON"].nunique())
print("Duplicados:", df_temperatura.duplicated(
    ["ID_POLIGON", "año", "mes", "variable"]
).sum())


1/96: Tmax_2022_01.tif
2/96: Tmax_2022_02.tif
3/96: Tmax_2022_03.tif
4/96: Tmax_2022_04.tif
5/96: Tmax_2022_05.tif
6/96: Tmax_2022_06.tif
7/96: Tmax_2022_07.tif
8/96: Tmax_2022_08.tif
9/96: Tmax_2022_09.tif
10/96: Tmax_2022_10.tif
11/96: Tmax_2022_11.tif
12/96: Tmax_2022_12.tif
13/96: Tmax_2023_01.tif
14/96: Tmax_2023_02.tif
15/96: Tmax_2023_03.tif
16/96: Tmax_2023_04.tif
17/96: Tmax_2023_05.tif
18/96: Tmax_2023_06.tif
19/96: Tmax_2023_07.tif
20/96: Tmax_2023_08.tif
21/96: Tmax_2023_09.tif
22/96: Tmax_2023_10.tif
23/96: Tmax_2023_11.tif
24/96: Tmax_2023_12.tif
25/96: Tmax_2024_01.tif
26/96: Tmax_2024_02.tif
27/96: Tmax_2024_03.tif
28/96: Tmax_2024_04.tif
29/96: Tmax_2024_05.tif
30/96: Tmax_2024_06.tif
31/96: Tmax_2024_07.tif
32/96: Tmax_2024_08.tif
33/96: Tmax_2024_09.tif
34/96: Tmax_2024_10.tif
35/96: Tmax_2024_11.tif
36/96: Tmax_2024_12.tif
37/96: Tmax_2025_01.tif
38/96: Tmax_2025_02.tif
39/96: Tmax_2025_03.tif
40/96: Tmax_2025_04.tif
41/96: Tmax_2025_05.tif
42/96: Tmax_2025_06.tif
4

In [32]:
display(df_temperatura.head())

,ID_POLIGON,año,mes,variable,temperatura_media,cobertura_pct,pixeles_validos
0,AGC_196,2022,1,Tmax,20.394972,100.0,1
1,AGC_044,2022,1,Tmax,20.436686,100.0,1
2,AGC_020,2022,1,Tmax,20.436686,100.0,1
3,AGC_174,2022,1,Tmax,20.226089,100.0,1
4,AGC_083,2022,1,Tmax,20.155764,100.0,1


In [33]:

# Separar las estadísticas por tipo de temperatura
print(df_temperatura.groupby("variable")["temperatura_media"].agg(["count", "mean", "std", "min", "median", "max"]).round(2))

# Comprobar valores faltantes
print("\nValores faltantes:")
print(df_temperatura[["temperatura_media", "cobertura_pct", "pixeles_validos"]].isna().sum())

# Comprobar cobertura por variable
print("\nCobertura mínima por variable:")
print(df_temperatura.groupby("variable")["cobertura_pct"].agg(["min", "mean", "max"]))

# Verificar que Tmax no sea menor que Tmin
temperaturas_ancho = df_temperatura.pivot(
    index=["ID_POLIGON", "año", "mes"],
    columns="variable",
    values="temperatura_media"
)

inconsistencias = (
    temperaturas_ancho["Tmax"] < temperaturas_ancho["Tmin"]
)

print("\nPares Tmax/Tmin:", len(temperaturas_ancho))
print("Casos Tmax < Tmin:", inconsistencias.sum())


          count   mean   std    min  median    max
variable                                          
Tmax       9456  23.49  2.18  18.76   23.35  30.16
Tmin       9456  11.23  2.54   5.42   11.55  16.26

Valores faltantes:
temperatura_media    0
cobertura_pct        0
pixeles_validos      0
dtype: int64

Cobertura mínima por variable:
            min   mean    max
variable                     
Tmax      100.0  100.0  100.0
Tmin      100.0  100.0  100.0

Pares Tmax/Tmin: 9456
Casos Tmax < Tmin: 0


In [34]:
df_temperatura_mensual = df_temperatura.pivot(index=['ID_POLIGON', 'año', 'mes'], columns='variable', values='temperatura_media').reset_index()

df_temperatura_mensual.columns.name = None

#Creamos la amplitud térmica
df_temperatura_mensual['amplitud_termica'] = df_temperatura_mensual['Tmax'] - df_temperatura_mensual['Tmin']

print("Dimensiones:", df_temperatura_mensual.shape)
print("Duplicados:", df_temperatura_mensual.duplicated(["ID_POLIGON", "año", "mes"]).sum())

print(df_temperatura_mensual[["Tmax", "Tmin", "amplitud_termica"]].describe().round(2))


Dimensiones: (9456, 6)
Duplicados: 0
          Tmax     Tmin  amplitud_termica
count  9456.00  9456.00           9456.00
mean     23.49    11.23             12.26
std       2.18     2.54              2.35
min      18.76     5.42              7.45
25%      21.98     9.09             10.18
50%      23.35    11.55             12.42
75%      25.05    13.34             14.14
max      30.16    16.26             17.81


### Precipitación + Clima

In [35]:
df_clima = df_precipitation.merge(df_temperatura_mensual, on=['ID_POLIGON', 'año', 'mes'], how='outer', validate='one_to_one')

print(f'Shape: {df_clima.shape}')
print(f'Parcelas Únicas: {df_clima.nunique()}')
print("Duplicados:", df_clima.duplicated(["ID_POLIGON", "año", "mes"]).sum())

print("\nValores faltantes:")
print(df_clima.isna().sum())

print(df_clima.head())


Shape: (9456, 9)
Parcelas Únicas: ID_POLIGON           197
año                    4
mes                   12
precipitacion_mm    2855
cobertura_pct         43
pixeles_validos        3
Tmax                2946
Tmin                3004
amplitud_termica    3272
dtype: int64
Duplicados: 0

Valores faltantes:
ID_POLIGON          0
año                 0
mes                 0
precipitacion_mm    0
cobertura_pct       0
pixeles_validos     0
Tmax                0
Tmin                0
amplitud_termica    0
dtype: int64
  ID_POLIGON   año  mes  precipitacion_mm  cobertura_pct  pixeles_validos  \
0    AGC_001  2022    1         19.545547          100.0                1   
1    AGC_001  2022    2         14.697198          100.0                1   
2    AGC_001  2022    3          4.974521          100.0                1   
3    AGC_001  2022    4         47.755169          100.0                1   
4    AGC_001  2022    5         45.257622          100.0                1   

        Tmax       T

In [36]:
df_clima_2025 = df_clima[df_clima['año'] == 2025].copy()

resumen_climatico_2025 = df_clima_2025.groupby('mes').agg( precipitacion_media=("precipitacion_mm", "mean"),
        precipitacion_std=("precipitacion_mm", "std"),
        Tmax_media=("Tmax", "mean"),
        Tmin_media=("Tmin", "mean"),
        amplitud_media=("amplitud_termica", "mean")).round(2)

print(resumen_climatico_2025)

     precipitacion_media  precipitacion_std  Tmax_media  Tmin_media  \
mes                                                                   
1                  17.40               3.67       20.93        7.80   
2                  35.99               8.36       22.62        9.54   
3                   8.73               1.57       25.46       10.45   
4                  12.16               2.78       25.92       10.75   
5                 137.92              37.23       26.21       13.78   
6                 245.94              51.41       22.63       14.75   
7                 148.53              27.29       22.93       13.21   
8                 131.91              25.74       23.56       13.35   
9                 161.99              52.52       22.59       13.45   
10                124.17              29.95       21.87       10.89   
11                 14.81               7.10       21.95        8.04   
12                 14.09               3.84       20.08        8.87   

     

In [37]:
df_clima_2025.plot(x='mes', y='precipitacion_media', kind='bar')
plt.show()

KeyError: 'precipitacion_media'

In [39]:
#Creamos ventanas temporlaes
ventanas = {
    'anual': list(range(1,13)),
    'abril_octubre': list(range(4, 11)),
    'mayo_octubre': list(range(5, 11)),
}

resultados_ventanas = []

for nombre_ventana, meses in ventanas.items():
    df_ventana = df_clima_2025[df_clima_2025['mes'].isin(meses)]
    
    resumen = df_ventana.groupby("ID_POLIGON").agg(precipitacion_acumulada=("precipitacion_mm", "sum"),
            Tmax_media=("Tmax", "mean"),
            Tmin_media=("Tmin", "mean"),
            amplitud_termica_media=("amplitud_termica", "mean")
            ,meses_disponibles=("mes", "nunique")).reset_index()

    resumen['ventana'] = nombre_ventana
    
    resultados_ventanas.append(resumen)
    
df_caracteristicas_climaticas = pd.concat(resultados_ventanas, ignore_index=True)


print("Dimensiones:", df_caracteristicas_climaticas.shape)
print("Parcelas únicas:",df_caracteristicas_climaticas["ID_POLIGON"].nunique())
print("Ventanas:",df_caracteristicas_climaticas["ventana"].unique())
print("Duplicados:",df_caracteristicas_climaticas.duplicated(["ID_POLIGON", "ventana"]).sum())
print("Valores faltantes:\n",df_caracteristicas_climaticas.isna().sum())

Dimensiones: (591, 7)
Parcelas únicas: 197
Ventanas: <StringArray>
['anual', 'abril_octubre', 'mayo_octubre']
Length: 3, dtype: str
Duplicados: 0
Valores faltantes:
 ID_POLIGON                 0
precipitacion_acumulada    0
Tmax_media                 0
Tmin_media                 0
amplitud_termica_media     0
meses_disponibles          0
ventana                    0
dtype: int64


In [40]:

columnas_resumen = [
    "precipitacion_acumulada",
    "Tmax_media",
    "Tmin_media",
    "amplitud_termica_media"
]

print(
    df_caracteristicas_climaticas
    .groupby("ventana")[columnas_resumen]
    .agg(["mean", "std", "min", "max"])
    .round(2)
)


              precipitacion_acumulada                          Tmax_media  \
                                 mean     std     min      max       mean   
ventana                                                                     
abril_octubre                  962.61  171.30  702.79  1459.14      23.67   
anual                         1053.63  190.86  766.09  1593.25      23.06   
mayo_octubre                   950.45  169.52  694.48  1435.35      23.30   

                                  Tmin_media                      \
                std    min    max       mean   std    min    max   
ventana                                                            
abril_octubre  0.16  23.25  24.01      12.88  0.57  11.42  13.48   
anual          0.12  22.72  23.27      11.24  0.61   9.85  11.91   
mayo_octubre   0.17  22.77  23.64      13.24  0.52  11.81  13.79   

              amplitud_termica_media                      
                                mean   std    min    max  
ventana   

- La precipitación se concentra en abril–octubre: esta ventana acumula, en promedio, 962.61 unidades, frente a 1,053.63 en todo el año.
- Los meses cálidos tienen noches más templadas: la Tmin media aumenta de 11.24 anual a 13.24 en mayo–octubre.
- La amplitud térmica disminuye: pasa de 11.82 anual a 10.06 en mayo–octubre, coherente con la diferencia entre las temperaturas de ambas estaciones.
Estas son diferencias climáticas descriptivas. Todavía no nos dicen qué ventana explica mejor el rendimiento

In [41]:

resumen_precipitacion = (
    df_caracteristicas_climaticas
    .groupby("ventana")["precipitacion_acumulada"]
    .agg(["mean", "std", "min", "median", "max"])
)

resumen_precipitacion["cv_pct"] = (
    100
    * resumen_precipitacion["std"]
    / resumen_precipitacion["mean"]
)

print(resumen_precipitacion.round(2))

                  mean     std     min   median      max  cv_pct
ventana                                                         
abril_octubre   962.61  171.30  702.79   961.35  1459.14   17.80
anual          1053.63  190.86  766.09  1053.32  1593.25   18.11
mayo_octubre    950.45  169.52  694.48   949.67  1435.35   17.84


Esto significa que la desviación estándar de la precipitación acumulada entre parcelas equivale aproximadamente al 17.8% de su media.
La interpretación importante: las tres ventanas presentan una variabilidad espacial relativa muy parecida. Ninguna destaca claramente sobre las otras en este aspecto.

Actualmente, df_caracteristicas_climaticas tiene 591 filas porque cada parcela aparece tres veces, una por ventana. Para combinarlo con topografía y satélite necesitamos que cada parcela ocupe una sola fila.
Vamos a convertir las ventanas en columnas. Ejecuta lo siguiente:

In [42]:
df_clima_ancho = (
    df_caracteristicas_climaticas
    .pivot(
        index="ID_POLIGON",
        columns="ventana",
        values=[
            "precipitacion_acumulada",
            "Tmax_media",
            "Tmin_media",
            "amplitud_termica_media"
        ]
    )
)

df_clima_ancho.columns = [
    f"{variable}_{ventana}"
    for variable, ventana in df_clima_ancho.columns
]

df_clima_ancho = df_clima_ancho.reset_index()

print("Dimensiones:", df_clima_ancho.shape)
print("Parcelas únicas:", df_clima_ancho["ID_POLIGON"].nunique())
print("Valores faltantes:", df_clima_ancho.isna().sum().sum())

display(df_clima_ancho.head())

Dimensiones: (197, 13)
Parcelas únicas: 197
Valores faltantes: 0


,ID_POLIGON,precipitacion_acumulada_abril_octubre,precipitacion_acumulada_anual,precipitacion_acumulada_mayo_octubre,Tmax_media_abril_octubre,Tmax_media_anual,Tmax_media_mayo_octubre,Tmin_media_abril_octubre,Tmin_media_anual,Tmin_media_mayo_octubre,amplitud_termica_media_abril_octubre,amplitud_termica_media_anual,amplitud_termica_media_mayo_octubre
0,AGC_001,1360.919123,1494.247836,1337.875435,23.349517,22.812208,22.893978,11.596077,9.978796,11.985211,11.753440,12.833412,10.908767
1,AGC_002,961.350809,1053.319641,949.671089,23.780726,23.157021,23.429427,13.328253,11.694942,13.664148,10.452473,11.462079,9.765279
2,AGC_003,1335.090225,1477.197298,1322.676590,23.801091,23.163777,23.455089,13.277613,11.593201,13.642318,10.523478,11.570577,9.812771
3,AGC_004,887.946089,966.084870,875.289787,23.791028,23.134377,23.400179,12.122947,10.367646,12.559611,11.668081,12.766731,10.840568
4,AGC_005,793.730323,868.425233,784.052917,23.513994,22.966181,23.135269,13.248211,11.715880,13.529122,10.265783,11.250301,9.606148


In [43]:
print("Clima:", df_clima_ancho.shape)
print("Topografía:", df_topografia.shape)

print("IDs únicos en clima:",
      df_clima_ancho["ID_POLIGON"].nunique())

print("IDs únicos en topografía:",
      df_topografia["ID_POLIGON"].nunique())

print("IDs de clima duplicados:",
      df_clima_ancho["ID_POLIGON"].duplicated().sum())

print("IDs de topografía duplicados:",
      df_topografia["ID_POLIGON"].duplicated().sum())

Clima: (197, 13)
Topografía: (197, 5)
IDs únicos en clima: 197
IDs únicos en topografía: 197
IDs de clima duplicados: 0
IDs de topografía duplicados: 0


### Topografía + Clima
En un mismo data frame juntamos los datos de clima con los de topografía

In [44]:
df_features = df_clima_ancho.merge(df_topografia, on='ID_POLIGON', how='inner', validate='one_to_one')

print('DF CARACTERÍSTICAS')
print(f'Shape: {df_features.shape}')
print(f'Unique Values: {df_features["ID_POLIGON"].nunique()}')
print("Valores faltantes:",
      df_features.isna().sum().sum())

DF CARACTERÍSTICAS
Shape: (197, 17)
Unique Values: 197
Valores faltantes: 0


Revisamos los datos que tienen falta de datos en pendeinte.

In [45]:
# Parcelas con menor cobertura del raster de pendiente
df_features[
    [
        "ID_POLIGON",
        "pendiente_media",
        "cobertura_pendiente_pct"
    ]
].sort_values(
    "cobertura_pendiente_pct"
).head(10)

,ID_POLIGON,pendiente_media,cobertura_pendiente_pct
169,AGC_170,1.391080,2.096897
22,AGC_023,1.766181,28.635263
163,AGC_164,0.903925,55.269931
19,AGC_020,0.561396,65.250345
43,AGC_044,0.348862,98.025585
114,AGC_115,0.360615,100.000000
94,AGC_095,4.145282,100.000000
63,AGC_064,6.216479,100.000000
24,AGC_025,0.871498,100.000000
145,AGC_146,0.394924,100.000000


In [46]:
df_features["cobertura_pendiente_pct"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50]
).round(2)

count    197.00
mean      98.73
std        9.47
min        2.10
1%        54.20
5%       100.00
10%      100.00
25%      100.00
50%      100.00
max      100.00
Name: cobertura_pendiente_pct, dtype: float64

In [47]:
df_features["pendiente_baja_cobertura"] = (
    df_features["cobertura_pendiente_pct"] < 90
).astype(int)

print(
    df_features[
        [
            "ID_POLIGON",
            "pendiente_media",
            "cobertura_pendiente_pct",
            "pendiente_baja_cobertura"
        ]
    ].query("pendiente_baja_cobertura == 1")
)

print(
    "\nParcelas con baja cobertura:",
    df_features["pendiente_baja_cobertura"].sum()
)

    ID_POLIGON  pendiente_media  cobertura_pendiente_pct  \
19     AGC_020         0.561396                65.250345   
22     AGC_023         1.766181                28.635263   
163    AGC_164         0.903925                55.269931   
169    AGC_170         1.391080                 2.096897   

     pendiente_baja_cobertura  
19                          1  
22                          1  
163                         1  
169                         1  

Parcelas con baja cobertura: 4


## Datos Satelitales

In [48]:
df_basic = pd.read_csv('../../DataSet/DATASET_RETO_AGRO_2026/Conjunto_datos_BASICO_AgroCebada2026.csv')
df_pro = pd.read_csv('../../DataSet/DATASET_RETO_AGRO_2026/Conjunto_datos_PRO_AgroCebada.csv')

In [49]:
for nombre, df in [('Basic', df_basic), ('Pro', df_pro)]:
    print(f'Dataset: {nombre}')
    print(f'Shape: {df.shape}')
    
    print('Columns: ')
    print(df.columns.tolist())
    
    print(f'Data types: {df.dtypes.value_counts()}')
    
    print('First 3: ')
    print(df.head(3))

Dataset: Basic
Shape: (107666, 96)
Columns: 
['ID_POLIGONO', 'fecha_captura', 'sensor', 'ndvi_promedio', 'ndvi_std', 'ndvi_max', 'ndvi_min', 'evi_promedio', 'evi_std', 'evi_max', 'evi_min', 'crc_promedio', 'crc_std', 'crc_max', 'crc_min', 'mcrc_promedio', 'mcrc_std', 'mcrc_max', 'mcrc_min', 'ndsvi_promedio', 'ndsvi_std', 'ndsvi_max', 'ndsvi_min', 'srndi_promedio', 'srndi_std', 'srndi_max', 'srndi_min', 'mra_promedio', 'mra_std', 'mra_max', 'mra_min', 'lai_promedio', 'lai_std', 'lai_max', 'lai_min', 'fapar_promedio', 'fapar_std', 'fapar_max', 'fapar_min', 'savi_promedio', 'savi_std', 'savi_max', 'savi_min', 'vi6t_promedio', 'vi6t_std', 'vi6t_max', 'vi6t_min', 'ndti_promedio', 'ndti_std', 'ndti_max', 'ndti_min', 'sti_promedio', 'sti_std', 'sti_max', 'sti_min', 'ndwi_promedio', 'ndwi_std', 'ndwi_max', 'ndwi_min', 'endwi_promedio', 'endwi_std', 'endwi_max', 'endwi_min', 'msi_promedio', 'msi_std', 'msi_max', 'msi_min', 'emsi_promedio', 'emsi_std', 'emsi_max', 'emsi_min', 'dswi2_promedio', '

In [50]:

# Trabajaremos sobre una copia
df_pro_trabajo = df_pro.copy()

# Convertir las fechas al formato datetime
df_pro_trabajo["fecha_captura"] = pd.to_datetime(
    df_pro_trabajo["fecha_captura"],
    format="%d/%m/%Y",
    errors="coerce"
)

# Crear variables temporales
df_pro_trabajo["año"] = df_pro_trabajo["fecha_captura"].dt.year
df_pro_trabajo["mes"] = df_pro_trabajo["fecha_captura"].dt.month

print("Fechas no interpretadas:",
      df_pro_trabajo["fecha_captura"].isna().sum())

print("\nRango de fechas:")
print(df_pro_trabajo["fecha_captura"].min(),
      "a",
      df_pro_trabajo["fecha_captura"].max())

print("\nObservaciones por mes:")
print(df_pro_trabajo.groupby("mes").size())

print("\nParcelas únicas:", df_pro_trabajo["ID_POLIGONO"].nunique())

# Disponibilidad de NDVI por mes
print("\nPorcentaje de NDVI válido por mes:")
print(
    (df_pro_trabajo.groupby("mes")["ndvi_promedio"]
     .apply(lambda x: 100 * x.notna().mean()))
    .round(2)
)


Fechas no interpretadas: 0

Rango de fechas:
2025-01-01 00:00:00 a 2025-12-30 00:00:00

Observaciones por mes:
mes
1     4316
2     4135
3     5166
4     4760
5     4436
6     2213
7     2940
8     4381
9     3964
10    3561
11    3407
12    4525
dtype: int64

Parcelas únicas: 197

Porcentaje de NDVI válido por mes:
mes
1     70.83
2     82.25
3     95.61
4     94.03
5     70.29
6     55.22
7     61.09
8     73.39
9     54.04
10    64.00
11    86.62
12    69.64
Name: ndvi_promedio, dtype: float64


#### Cobertura por parcel y mes.
El objetivo es construir una tabla con una fila por parcela para predecir el rendimiento. Para lograrlo, tendremos que resumir las observaciones temporales de Planet en variables como:
- NDVI medio durante una ventana temporal.
- NDVI máximo, como indicador del verdor máximo observado.
- Variabilidad del NDVI durante el periodo.
- EVI, LAI y MSAVI resumidos de manera equivalente.

In [51]:

# Número de observaciones por parcela y mes
observaciones_mes = (
    df_pro_trabajo
    .groupby(["ID_POLIGONO", "mes"])
    .size()
    .rename("n_observaciones")
    .reset_index()
)

print("Observaciones por parcela y mes:")
print(
    observaciones_mes["n_observaciones"]
    .describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95])
)

# Resumen de la cantidad de observaciones por mes
resumen_observaciones = (
    observaciones_mes
    .groupby("mes")["n_observaciones"]
    .agg(["count", "min", "median", "mean", "max"])
)

print("\nResumen por mes:")
print(resumen_observaciones.round(2))

# Cobertura de NDVI por parcela y mes
cobertura_ndvi = (
    df_pro_trabajo
    .groupby(["ID_POLIGONO", "mes"])["ndvi_promedio"]
    .agg(
        n_total="size",
        n_validos="count"
    )
    .reset_index()
)

cobertura_ndvi["porcentaje_valido"] = (
    100 * cobertura_ndvi["n_validos"] / cobertura_ndvi["n_total"]
)

print("\nCobertura de NDVI por parcela y mes:")
print(
    cobertura_ndvi["porcentaje_valido"]
    .describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95])
)

# ¿Cuántas parcelas tienen al menos una observación válida por mes?
parcelas_con_ndvi = (
    cobertura_ndvi
    .assign(tiene_ndvi=cobertura_ndvi["n_validos"] > 0)
    .groupby("mes")["tiene_ndvi"]
    .sum()
)

print("\nParcelas con al menos un NDVI válido por mes:")
print(parcelas_con_ndvi)


Observaciones por parcela y mes:
count    2364.000000
mean       20.221658
std         4.362337
min         8.000000
5%         11.000000
25%        17.000000
50%        21.000000
75%        23.000000
95%        26.000000
max        30.000000
Name: n_observaciones, dtype: float64

Resumen por mes:
     count  min  median   mean  max
mes                                
1      197   20    22.0  21.91   24
2      197   16    23.0  20.99   24
3      197   23    27.0  26.22   30
4      197   19    26.0  24.16   26
5      197   20    23.0  22.52   24
6      197    8    11.0  11.23   16
7      197   13    15.0  14.92   17
8      197   18    23.0  22.24   24
9      197   18    19.0  20.12   24
10     197   14    17.0  18.08   22
11     197   14    17.0  17.29   19
12     197   19    23.0  22.97   27

Cobertura de NDVI por parcela y mes:
count    2364.000000
mean       73.051522
std        15.882245
min        23.809524
5%         45.454545
25%        62.500000
50%        72.727273
75%        8

In [52]:

indices_pro = [
    "ndvi", "evi", "lai", "msavi"
]

# Porcentaje de valores válidos en cada índice
resumen_indices = pd.DataFrame({
    indice: {
        "porcentaje_valido": (
            100 * df_pro_trabajo[f"{indice}_promedio"].notna().mean()
        ),
        "media": df_pro_trabajo[f"{indice}_promedio"].mean(),
        "desviacion": df_pro_trabajo[f"{indice}_promedio"].std(),
        "minimo": df_pro_trabajo[f"{indice}_promedio"].min(),
        "maximo": df_pro_trabajo[f"{indice}_promedio"].max()
    }
    for indice in indices_pro
}).T

print(resumen_indices.round(3))

# Disponibilidad mensual de cada índice
disponibilidad_mensual = (
    df_pro_trabajo
    .groupby("mes")[
        [f"{indice}_promedio" for indice in indices_pro]
    ]
    .agg(lambda columna: 100 * columna.notna().mean())
)

print("\nPorcentaje válido por mes e índice:")
print(disponibilidad_mensual.round(2))


       porcentaje_valido  media  desviacion  minimo  maximo
ndvi              74.778  0.354       0.175   0.007   0.907
evi               74.778  0.215       0.123   0.028   0.877
lai               74.778  1.215       0.613  -0.112   3.663
msavi             74.778  0.211       0.123   0.010   0.780

Porcentaje válido por mes e índice:
     ndvi_promedio  evi_promedio  lai_promedio  msavi_promedio
mes                                                           
1            70.83         70.83         70.83           70.83
2            82.25         82.25         82.25           82.25
3            95.61         95.61         95.61           95.61
4            94.03         94.03         94.03           94.03
5            70.29         70.29         70.29           70.29
6            55.22         55.22         55.22           55.22
7            61.09         61.09         61.09           61.09
8            73.39         73.39         73.39           73.39
9            54.04         54.04 

### Auditamos estructura.


In [54]:
df_rendimiento = pd.read_csv('../../DataSet/DATASET_RETO_AGRO_2026/ID_area_rendimiento_70_30_Reto_AgroCebada.csv')

In [55]:

tablas = {
    "parcelas": parcelas_gdf,
    "rendimiento": df_rendimiento,
    "basico": df_basic,
    "pro": df_pro_trabajo,
    "clima_mensual": df_clima,
    "clima_agregado": df_clima_ancho,
    "topografia": df_topografia,
    "caracteristicas": df_features
}

for nombre, df in tablas.items():
    print(f"\n--- {nombre} ---")
    print("Dimensiones:", df.shape)
    print("ID disponible:", "ID_POLIGON" in df.columns
          or "ID_POLIGONO" in df.columns)

    columna_id = (
        "ID_POLIGON"
        if "ID_POLIGON" in df.columns
        else "ID_POLIGONO"
        if "ID_POLIGONO" in df.columns
        else None
    )

    if columna_id:
        print("IDs únicos:", df[columna_id].nunique())
        print("IDs nulos:", df[columna_id].isna().sum())



--- parcelas ---
Dimensiones: (197, 7)
ID disponible: True
IDs únicos: 197
IDs nulos: 0

--- rendimiento ---
Dimensiones: (197, 5)
ID disponible: True
IDs únicos: 197
IDs nulos: 0

--- basico ---
Dimensiones: (107666, 96)
ID disponible: True
IDs únicos: 197
IDs nulos: 0

--- pro ---
Dimensiones: (47804, 22)
ID disponible: True
IDs únicos: 197
IDs nulos: 0

--- clima_mensual ---
Dimensiones: (9456, 9)
ID disponible: True
IDs únicos: 197
IDs nulos: 0

--- clima_agregado ---
Dimensiones: (197, 13)
ID disponible: True
IDs únicos: 197
IDs nulos: 0

--- topografia ---
Dimensiones: (197, 5)
ID disponible: True
IDs únicos: 197
IDs nulos: 0

--- caracteristicas ---
Dimensiones: (197, 18)
ID disponible: True
IDs únicos: 197
IDs nulos: 0


In [56]:

# Comprobar nombres de columnas temporales
print("Columnas PRO:")
print(df_pro_trabajo.columns.tolist())

print("\nColumnas Básico:")
print(df_basic.columns.tolist())

# Duplicados en Planet PRO por parcela y fecha
duplicados_pro = df_pro_trabajo.duplicated(
    subset=["ID_POLIGONO", "fecha_captura"],
    keep=False
)

print("\nPRO:")
print("Filas involucradas en duplicados:", duplicados_pro.sum())

# Duplicados en Dataset Básico por parcela, fecha y sensor
duplicados_basico = df_basic.duplicated(
    subset=["ID_POLIGONO", "fecha_captura", "sensor"],
    keep=False
)

print("\nBásico:")
print("Filas involucradas en duplicados:", duplicados_basico.sum())


Columnas PRO:
['ID_POLIGONO', 'fecha_captura', 'sensor', 'ndvi_promedio', 'ndvi_std', 'ndvi_max', 'ndvi_min', 'evi_promedio', 'evi_std', 'evi_max', 'evi_min', 'lai_promedio', 'lai_std', 'lai_max', 'lai_min', 'msavi_promedio', 'msavi_std', 'msavi_max', 'msavi_min', 'porcentaje_nubosidad', 'año', 'mes']

Columnas Básico:
['ID_POLIGONO', 'fecha_captura', 'sensor', 'ndvi_promedio', 'ndvi_std', 'ndvi_max', 'ndvi_min', 'evi_promedio', 'evi_std', 'evi_max', 'evi_min', 'crc_promedio', 'crc_std', 'crc_max', 'crc_min', 'mcrc_promedio', 'mcrc_std', 'mcrc_max', 'mcrc_min', 'ndsvi_promedio', 'ndsvi_std', 'ndsvi_max', 'ndsvi_min', 'srndi_promedio', 'srndi_std', 'srndi_max', 'srndi_min', 'mra_promedio', 'mra_std', 'mra_max', 'mra_min', 'lai_promedio', 'lai_std', 'lai_max', 'lai_min', 'fapar_promedio', 'fapar_std', 'fapar_max', 'fapar_min', 'savi_promedio', 'savi_std', 'savi_max', 'savi_min', 'vi6t_promedio', 'vi6t_std', 'vi6t_max', 'vi6t_min', 'ndti_promedio', 'ndti_std', 'ndti_max', 'ndti_min', 'sti

#### Distribución temporal.

In [57]:

# Distribución de observaciones por año y sensor
# Crear variables temporales
df_basico = df_basic.copy()

df_basico["fecha_captura"] = pd.to_datetime(
    df_basico["fecha_captura"],
    errors="coerce"
)

df_basico["año"] = df_basico["fecha_captura"].dt.year
df_basico["mes"] = df_basico["fecha_captura"].dt.month


resumen_sensor = (
    df_basico
    .groupby(["año", "sensor"])
    .agg(
        observaciones=("ID_POLIGONO", "size"),
        parcelas=("ID_POLIGONO", "nunique"),
        ndvi_validos=("ndvi_promedio", "count"),
        nubosidad_media=("porcentaje_nubosidad", "mean")
    )
    .reset_index()
)

resumen_sensor["ndvi_valido_pct"] = (
    100 * resumen_sensor["ndvi_validos"]
    / resumen_sensor["observaciones"]
)

print(resumen_sensor.round(2).to_string(index=False))


C:\Users\gonzo\AppData\Local\Temp\ipykernel_24000\3104702767.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_basico["fecha_captura"] = pd.to_datetime(


 año     sensor  observaciones  parcelas  ndvi_validos  nubosidad_media  ndvi_valido_pct
2022    Landsat          12806       197             0            39.49             0.00
2022 Sentinel-2          13922       197         10045            33.58            72.15
2023    Landsat          13125       197             0            37.64             0.00
2023 Sentinel-2          13917       197         10301            31.22            74.02
2024    Landsat          11759       197             0            33.82             0.00
2024 Sentinel-2          14381       197         11440            25.51            79.55
2025    Landsat          10936       197             0            33.27             0.00
2025 Sentinel-2          16820       197         13091            27.03            77.83


In [58]:
resumen_mensual = (
    df_basico
    .groupby(["sensor", "mes"])
    .agg(
        observaciones=("ID_POLIGONO", "size"),
        ndvi_validos=("ndvi_promedio", "count"),
        parcelas=("ID_POLIGONO", "nunique")
    )
    .reset_index()
)

resumen_mensual["ndvi_valido_pct"] = (
    100 * resumen_mensual["ndvi_validos"]
    / resumen_mensual["observaciones"]
)

print(resumen_mensual.round(2).to_string(index=False))

    sensor  mes  observaciones  ndvi_validos  parcelas  ndvi_valido_pct
   Landsat    1           3734             0       197             0.00
   Landsat    2           4076             0       197             0.00
   Landsat    3           3611             0       197             0.00
   Landsat    4           4290             0       197             0.00
   Landsat    5           4319             0       197             0.00
   Landsat    6           4183             0       197             0.00
   Landsat    7           3912             0       197             0.00
   Landsat    8           4250             0       197             0.00
   Landsat    9           3866             0       197             0.00
   Landsat   10           4075             0       197             0.00
   Landsat   11           4183             0       197             0.00
   Landsat   12           4127             0       197             0.00
Sentinel-2    1           5319          4293       197          

In [59]:
cobertura_anual_mensual = (
    df_basico[df_basico["sensor"] == "Sentinel-2"]
    .groupby(["año", "mes"])
    .agg(
        observaciones=("ID_POLIGONO", "size"),
        ndvi_validos=("ndvi_promedio", "count")
    )
    .reset_index()
)

cobertura_anual_mensual["ndvi_valido_pct"] = (
    100 * cobertura_anual_mensual["ndvi_validos"]
    / cobertura_anual_mensual["observaciones"]
)

print(
    cobertura_anual_mensual
    .round(2)
    .to_string(index=False)
)

 año  mes  observaciones  ndvi_validos  ndvi_valido_pct
2022    1           1379          1060            76.87
2022    2            985           918            93.20
2022    3           1182           970            82.06
2022    4           1117           890            79.68
2022    5           1379          1109            80.42
2022    6           1182           794            67.17
2022    7           1182           787            66.58
2022    8           1182           549            46.45
2022    9           1182           261            22.08
2022   10           1182           785            66.41
2022   11            985           942            95.63
2022   12            985           980            99.49
2023    1           1379          1349            97.82
2023    2            985           926            94.01
2023    3           1182          1165            98.56
2023    4           1182           825            69.80
2023    5           1147           914          

In [60]:
cobertura_parcela_2025 = (
    df_basico[
        (df_basico["sensor"] == "Sentinel-2") &
        (df_basico["año"] == 2025)
    ]
    .groupby("ID_POLIGONO")
    .agg(
        observaciones=("ndvi_promedio", "size"),
        ndvi_validos=("ndvi_promedio", "count")
    )
    .reset_index()
)

cobertura_parcela_2025["ndvi_valido_pct"] = (
    100 * cobertura_parcela_2025["ndvi_validos"]
    / cobertura_parcela_2025["observaciones"]
)

print(
    cobertura_parcela_2025["ndvi_valido_pct"]
    .describe()
    .round(2)
)

print("\nParcelas con menor cobertura:")
print(
    cobertura_parcela_2025
    .sort_values("ndvi_valido_pct")
    .head(10)
    .round(2)
    .to_string(index=False)
)

count    197.00
mean      77.86
std        4.03
min       67.44
25%       75.58
50%       77.91
75%       80.23
max       85.71
Name: ndvi_valido_pct, dtype: float64

Parcelas con menor cobertura:
ID_POLIGONO  observaciones  ndvi_validos  ndvi_valido_pct
    AGC_049             86            58            67.44
    AGC_135             86            58            67.44
    AGC_127             86            59            68.60
    AGC_006             86            59            68.60
    AGC_111             86            59            68.60
    AGC_108             86            59            68.60
    AGC_132             86            59            68.60
    AGC_126             86            60            69.77
    AGC_013             86            60            69.77
    AGC_190             86            61            70.93


Tenemos muchas observaciones por parcela, pero el objetivo es terminar con una fila por parcela. Necesitamos resumir la evolución temporal sin perder toda la información estacional. 

In [61]:
df_s2_2025 = df_basico[
    (df_basico["sensor"] == "Sentinel-2") &
    (df_basico["año"] == 2025)
].copy()

resumen_ndvi_mensual = (
    df_s2_2025
    .groupby(["ID_POLIGONO", "mes"])
    .agg(
        ndvi_medio=("ndvi_promedio", "mean"),
        ndvi_std=("ndvi_promedio", "std"),
        ndvi_max=("ndvi_promedio", "max"),
        ndvi_min=("ndvi_promedio", "min"),
        observaciones=("ndvi_promedio", "size"),
        ndvi_validos=("ndvi_promedio", "count")
    )
    .reset_index()
)

print("Dimensiones:", resumen_ndvi_mensual.shape)
print(resumen_ndvi_mensual.head(15).round(3).to_string(index=False))

Dimensiones: (2364, 8)
ID_POLIGONO  mes  ndvi_medio  ndvi_std  ndvi_max  ndvi_min  observaciones  ndvi_validos
    AGC_001    1       0.142     0.015     0.157     0.125              6             4
    AGC_001    2       0.126     0.006     0.132     0.117              5             4
    AGC_001    3       0.129     0.007     0.139     0.122              8             7
    AGC_001    4       0.123     0.009     0.135     0.107              8             8
    AGC_001    5       0.108     0.023     0.123     0.073              7             4
    AGC_001    6       0.162     0.008     0.167     0.156              8             2
    AGC_001    7       0.246     0.054     0.351     0.210              7             6
    AGC_001    8       0.421     0.163     0.662     0.246              7             7
    AGC_001    9       0.594     0.143     0.745     0.336              6             6
    AGC_001   10       0.545     0.148     0.689     0.251              7             7
    AGC_0

Observamos que el NDVI medio aumenta considerablemente de junio a septiembre y disminuye en octubre. Esto describe la evolución de la señal de vegetación de esa parcela, aunque por sí solo no demuestra cuál fue su etapa fenológica ni cuál será su rendimiento.
Hay otra observación importante: en junio solo tenemos 2 valores válidos. El promedio de ese mes es, por tanto, menos robusto que el de octubre, que tiene 7.

In [63]:
# 1. ¿Cuántos grupos tienen NDVI medio ausente?
print(
    "Grupos con NDVI medio ausente:",
    resumen_ndvi_mensual["ndvi_medio"].isna().sum()
)

# 2. ¿Cuántos grupos tienen menos de 3 valores válidos?
print(
    "Grupos con menos de 3 NDVI válidos:",
    (resumen_ndvi_mensual["ndvi_validos"] < 3).sum()
)

# 3. Resumen de la cantidad de NDVI válidos por grupo
print(
    resumen_ndvi_mensual["ndvi_validos"]
    .describe()
    .round(2)
)

Grupos con NDVI medio ausente: 0
Grupos con menos de 3 NDVI válidos: 37
count    2364.00
mean        5.54
std         1.58
min         1.00
25%         4.00
50%         5.00
75%         7.00
max         8.00
Name: ndvi_validos, dtype: float64


Ahora vamos a crear una tabla ancha con las variables mensuales de NDVI medio y otra con el número de observaciones válidas. Así mantenemos tanto la señal de vegetación como información sobre su cobertura.

In [64]:
# A. Promedio mensual de NDVI: una columna por mes
ndvi_mensual_ancho = (
    resumen_ndvi_mensual
    .pivot(
        index="ID_POLIGONO",
        columns="mes",
        values="ndvi_medio"
    )
)

ndvi_mensual_ancho.columns = [
    f"ndvi_medio_mes_{mes:02d}"
    for mes in ndvi_mensual_ancho.columns
]

ndvi_mensual_ancho = ndvi_mensual_ancho.reset_index()


# B. Número de observaciones válidas por mes
ndvi_cobertura_ancho = (
    resumen_ndvi_mensual
    .pivot(
        index="ID_POLIGONO",
        columns="mes",
        values="ndvi_validos"
    )
)

ndvi_cobertura_ancho.columns = [
    f"ndvi_validos_mes_{mes:02d}"
    for mes in ndvi_cobertura_ancho.columns
]

ndvi_cobertura_ancho = ndvi_cobertura_ancho.reset_index()


# C. Revisar dimensiones y estructura
print("NDVI mensual:", ndvi_mensual_ancho.shape)
print("Cobertura mensual:", ndvi_cobertura_ancho.shape)

print(ndvi_mensual_ancho.head().round(3).to_string(index=False))

NDVI mensual: (197, 13)
Cobertura mensual: (197, 13)
ID_POLIGONO  ndvi_medio_mes_01  ndvi_medio_mes_02  ndvi_medio_mes_03  ndvi_medio_mes_04  ndvi_medio_mes_05  ndvi_medio_mes_06  ndvi_medio_mes_07  ndvi_medio_mes_08  ndvi_medio_mes_09  ndvi_medio_mes_10  ndvi_medio_mes_11  ndvi_medio_mes_12
    AGC_001              0.142              0.126              0.129              0.123              0.108              0.162              0.246              0.421              0.594              0.545              0.349              0.168
    AGC_002              0.143              0.160              0.161              0.144              0.148              0.365              0.670              0.624              0.459              0.402              0.230              0.181
    AGC_003              0.450              0.452              0.275              0.234              0.156              0.176              0.408              0.643              0.589              0.517              0.441       

In [65]:
# Comprobar identificadores únicos
print(
    "Parcelas únicas en NDVI:",
    ndvi_mensual_ancho["ID_POLIGONO"].nunique()
)

print(
    "Parcelas únicas en cobertura:",
    ndvi_cobertura_ancho["ID_POLIGONO"].nunique()
)

# Comprobar valores ausentes en los promedios mensuales
print(
    "\nValores ausentes en NDVI mensual:",
    ndvi_mensual_ancho.drop(columns="ID_POLIGONO").isna().sum().sum()
)

# Revisar cobertura mensual
print("\nObservaciones válidas por mes:")
print(
    ndvi_cobertura_ancho.drop(columns="ID_POLIGONO")
    .describe()
    .round(2)
)

Parcelas únicas en NDVI: 197
Parcelas únicas en cobertura: 197

Valores ausentes en NDVI mensual: 0

Observaciones válidas por mes:
       ndvi_validos_mes_01  ndvi_validos_mes_02  ndvi_validos_mes_03  \
count               197.00               197.00               197.00   
mean                  4.68                 3.85                 7.22   
std                   0.48                 0.36                 0.51   
min                   3.00                 3.00                 6.00   
25%                   4.00                 4.00                 7.00   
50%                   5.00                 4.00                 7.00   
75%                   5.00                 4.00                 8.00   
max                   5.00                 4.00                 8.00   

       ndvi_validos_mes_04  ndvi_validos_mes_05  ndvi_validos_mes_06  \
count               197.00               197.00               197.00   
mean                  7.77                 4.62                 3.20   
std

In [66]:
print("Dimensiones:", df_clima.shape)
print("Columnas:", df_clima.columns.tolist())

print(
    df_clima[df_clima["año"] == 2025]
    .head()
    .to_string(index=False)
)

Dimensiones: (9456, 9)
Columnas: ['ID_POLIGON', 'año', 'mes', 'precipitacion_mm', 'cobertura_pct', 'pixeles_validos', 'Tmax', 'Tmin', 'amplitud_termica']
ID_POLIGON  año  mes  precipitacion_mm  cobertura_pct  pixeles_validos      Tmax      Tmin  amplitud_termica
   AGC_001 2025    1         29.093111          100.0                1 20.859745  6.540424         14.319321
   AGC_001 2025    2         61.019646          100.0                1 22.330593  8.316397         14.014196
   AGC_001 2025    3          7.844304          100.0                1 25.099869  9.426901         15.672968
   AGC_001 2025    4         23.043688          100.0                1 26.082750  9.261271         16.821480
   AGC_001 2025    5        196.236435          100.0                1 25.510502 12.582640         12.927862


In [67]:
print(resumen_climatico_2025.head())

     precipitacion_media  precipitacion_std  Tmax_media  Tmin_media  \
mes                                                                   
1                  17.40               3.67       20.93        7.80   
2                  35.99               8.36       22.62        9.54   
3                   8.73               1.57       25.46       10.45   
4                  12.16               2.78       25.92       10.75   
5                 137.92              37.23       26.21       13.78   

     amplitud_media  
mes                  
1             13.13  
2             13.08  
3             15.01  
4             15.17  
5             12.43  


In [68]:
print("Dimensiones:", df_features.shape)

print(
    "Parcelas únicas:",
    df_features["ID_POLIGON"].nunique()
)

print(
    "IDs duplicados:",
    df_features["ID_POLIGON"].duplicated().sum()
)

print(df_features.columns.tolist())

Dimensiones: (197, 18)
Parcelas únicas: 197
IDs duplicados: 0
['ID_POLIGON', 'precipitacion_acumulada_abril_octubre', 'precipitacion_acumulada_anual', 'precipitacion_acumulada_mayo_octubre', 'Tmax_media_abril_octubre', 'Tmax_media_anual', 'Tmax_media_mayo_octubre', 'Tmin_media_abril_octubre', 'Tmin_media_anual', 'Tmin_media_mayo_octubre', 'amplitud_termica_media_abril_octubre', 'amplitud_termica_media_anual', 'amplitud_termica_media_mayo_octubre', 'elevacion_media', 'cobertura_elevacion_pct', 'pendiente_media', 'cobertura_pendiente_pct', 'pendiente_baja_cobertura']


### Unión para un solo DF.

In [69]:
ANIO = 2025
MES_INI, MES_FIN = 4, 10   
MIN_VALIDOS = 3            # mínimo de observaciones válidas para calcular estadísticas de un índice
MAX_NUBES = 70           # p. ej. 60 para descartar observaciones con más de 60 % de nubosidad


df_basico = df_basic.copy()
df_basico["fecha_captura"] = pd.to_datetime(df_basico["fecha_captura"], format="%d/%m/%Y", errors="coerce")
df_planet = df_pro.copy()
df_planet["fecha_captura"] = pd.to_datetime(df_planet["fecha_captura"], format="%d/%m/%Y", errors="coerce")

print("Fechas no interpretadas -> Básico:", df_basico["fecha_captura"].isna().sum(),
      "| PRO:", df_planet["fecha_captura"].isna().sum())

# % de valores no nulos
cols_prom = [c for c in df_basico.columns if c.endswith("_promedio")]
disponibilidad = (
    df_basico.groupby("sensor")[cols_prom]
    .apply(lambda g: 100 * g.notna().mean())
    .T.round(1)
)
print(disponibilidad)

Fechas no interpretadas -> Básico: 0 | PRO: 0
sensor          Landsat  Sentinel-2
ndvi_promedio       0.0        76.0
evi_promedio        0.0        76.0
crc_promedio        0.0        76.0
mcrc_promedio       0.0        76.0
ndsvi_promedio      0.0        76.0
srndi_promedio      0.0        76.0
mra_promedio        0.0        76.0
lai_promedio        0.0        76.0
fapar_promedio      0.0        76.0
savi_promedio       0.0        76.0
vi6t_promedio      63.9         0.0
ndti_promedio       0.0        76.0
sti_promedio        0.0        76.0
ndwi_promedio       0.0        76.0
endwi_promedio      0.0        76.0
msi_promedio        0.0        76.0
emsi_promedio       0.0        76.0
dswi2_promedio      0.0        76.0
dswi3_promedio      0.0        76.0
dswi4_promedio      0.0        76.0
dswi5_promedio      0.0        76.0
nddi_promedio       0.0        76.0
enddi_promedio      0.0        76.0


In [70]:
def _ventana(df, sensor=None):
    #Filtra sensor, año, meses del ciclo y (opcional) nubosidad.
    d = df
    if sensor is not None:
        d = d[d["sensor"] == sensor]
    d = d[(d["fecha_captura"].dt.year == ANIO) &
          d["fecha_captura"].dt.month.between(MES_INI, MES_FIN)]
    if MAX_NUBES is not None:
        d = d[d["porcentaje_nubosidad"] <= MAX_NUBES]
    return d


def resumen_temporal(df, indices, prefijo, sensor=None):
    #Una fila por parcela con estadísticos de la curva de cada índice.
    trapz = getattr(np, "trapezoid", None) or np.trapz
    filas = []
    for pid, g in _ventana(df, sensor).groupby("ID_POLIGONO"):
        fila = {"ID_POLIGON": pid}
        for idx in indices:
            col = f"{idx}_promedio"
            s = g[["fecha_captura", col]].dropna().sort_values("fecha_captura")
            fila[f"{prefijo}_{idx}_n_validos"] = len(s)
            if len(s) < MIN_VALIDOS:
                continue  # queda NaN: mejor que un promedio con 1-2 puntos
            x = s["fecha_captura"].dt.dayofyear.to_numpy(float)
            y = s[col].to_numpy(float)
            span = x[-1] - x[0]
            fila[f"{prefijo}_{idx}_media"] = y.mean()
            fila[f"{prefijo}_{idx}_std"] = y.std(ddof=1)
            fila[f"{prefijo}_{idx}_max"] = y.max()
            fila[f"{prefijo}_{idx}_doy_max"] = x[y.argmax()]          # día del año del pico
            fila[f"{prefijo}_{idx}_auc"] = trapz(y, x) / span if span > 0 else y.mean()
        filas.append(fila)
    return pd.DataFrame(filas)


def ndvi_mensual(df, prefijo, sensor=None):
    d = _ventana(df, sensor).copy()
    d["mes"] = d["fecha_captura"].dt.month
    m = d.groupby(["ID_POLIGONO", "mes"])["ndvi_promedio"].mean().unstack("mes")
    m.columns = [f"{prefijo}_ndvi_mes_{mes:02d}" for mes in m.columns]
    return m.reset_index().rename(columns={"ID_POLIGONO": "ID_POLIGON"})


s2   = resumen_temporal(df_basico, ["ndvi", "evi", "lai", "ndwi", "msi"], "s2", sensor="Sentinel-2")
pl   = resumen_temporal(df_planet, ["ndvi", "evi", "lai", "msavi"], "pl")
s2_m = ndvi_mensual(df_basico, "s2", sensor="Sentinel-2")
pl_m = ndvi_mensual(df_planet, "pl")

for nombre, d in [("S2 resumen", s2), ("PRO resumen", pl), ("S2 mensual", s2_m), ("PRO mensual", pl_m)]:
    print(f"{nombre}: {d.shape} | parcelas únicas: {d['ID_POLIGON'].nunique()}")

S2 resumen: (197, 31) | parcelas únicas: 197
PRO resumen: (197, 25) | parcelas únicas: 197
S2 mensual: (197, 8) | parcelas únicas: 197
PRO mensual: (197, 8) | parcelas únicas: 197


In [71]:
def indice_mensual(df, indice, prefijo, sensor=None):
    #Media mensual de un índice dentro del ciclo (una columna por mes)
    d = _ventana(df, sensor).copy()
    d["mes"] = d["fecha_captura"].dt.month
    m = d.groupby(["ID_POLIGONO", "mes"])[f"{indice}_promedio"].mean().unstack("mes")
    m.columns = [f"{prefijo}_{indice}_mes_{mes:02d}" for mes in m.columns]
    return m.reset_index().rename(columns={"ID_POLIGONO": "ID_POLIGON"})

ls   = resumen_temporal(df_basico, ["vi6t"], "ls", sensor="Landsat")
ls_m = indice_mensual(df_basico, "vi6t", "ls", sensor="Landsat")

print("Landsat resumen:", ls.shape, "| mensual:", ls_m.shape)
print(ls["ls_vi6t_n_validos"].describe().round(1))
print("Parcelas con menos de MIN_VALIDOS:", (ls["ls_vi6t_n_validos"] < MIN_VALIDOS).sum())

Landsat resumen: (197, 7) | mensual: (197, 8)
count    197.0
mean      18.8
std        5.0
min       11.0
25%       15.0
50%       17.0
75%       21.0
max       36.0
Name: ls_vi6t_n_validos, dtype: float64
Parcelas con menos de MIN_VALIDOS: 0


In [72]:
print(df_rendimiento.columns.tolist())
print(df_rendimiento.head())

col_id = next(c for c in df_rendimiento.columns if c.lower().startswith("id"))
col_y  = next(c for c in df_rendimiento.columns if "rend" in c.lower())
print("ID:", col_id, "| Rendimiento:", col_y)

df_y = (df_rendimiento[[col_id, col_y]]
        .rename(columns={col_id: "ID_POLIGON", col_y: "rendimiento_t_ha"}))

# Conjunto (ENTRENAMIENTO / PREDICCION), área y municipio vienen del shapefile
df_meta = parcelas_gdf[["ID_POLIGON", "CONJUNTO", "área_ha", "Municipio"]].copy()

['ID_POLIGONO', 'AREA_HA', 'RENDIMIENTO_T_HA', 'CONJUNTO', 'Unnamed: 4']
  ID_POLIGONO  AREA_HA  RENDIMIENTO_T_HA       CONJUNTO  Unnamed: 4
0     AGC_001     7.61              3.50  ENTRENAMIENTO         NaN
1     AGC_002     2.31              4.63  ENTRENAMIENTO         NaN
2     AGC_003    18.68               NaN     PREDICCION         NaN
3     AGC_004     8.00              3.00  ENTRENAMIENTO         NaN
4     AGC_005     2.27              5.16  ENTRENAMIENTO         NaN
ID: ID_POLIGONO | Rendimiento: RENDIMIENTO_T_HA


In [73]:
df_modelo = (
    df_features
    .merge(df_meta, on="ID_POLIGON", how="left", validate="one_to_one")
    .merge(df_y,    on="ID_POLIGON", how="left", validate="one_to_one")
    .merge(s2,      on="ID_POLIGON", how="left", validate="one_to_one")
    .merge(s2_m,    on="ID_POLIGON", how="left", validate="one_to_one")
    .merge(pl,      on="ID_POLIGON", how="left", validate="one_to_one")
    .merge(pl_m,    on="ID_POLIGON", how="left", validate="one_to_one")
    .merge(ls,      on="ID_POLIGON", how="left", validate="one_to_one")
    .merge(ls_m,    on="ID_POLIGON", how="left", validate="one_to_one")
)

assert len(df_modelo) == 197 and df_modelo["ID_POLIGON"].is_unique
print("Shape:", df_modelo.shape)

# El rendimiento debe existir en las 138 de entrenamiento y estar vacío en las 59 de predicción
print(df_modelo.groupby("CONJUNTO")["rendimiento_t_ha"].agg(filas="size", con_rendimiento="count"))

# Valores faltantes por columna (solo las que tienen)
faltantes = df_modelo.isna().mean().mul(100).round(1)
print(faltantes[faltantes > 0].sort_values(ascending=False))

Shape: (197, 103)
               filas  con_rendimiento
CONJUNTO                             
ENTRENAMIENTO    138              138
PREDICCION        59                0
rendimiento_t_ha    29.9
ls_vi6t_mes_09       7.1
ls_vi6t_mes_07       2.5
ls_vi6t_mes_06       1.0
dtype: float64


In [74]:
df_modelo.head()

,ID_POLIGON,precipitacion_acumulada_abril_octubre,precipitacion_acumulada_anual,precipitacion_acumulada_mayo_octubre,Tmax_media_abril_octubre,Tmax_media_anual,Tmax_media_mayo_octubre,Tmin_media_abril_octubre,Tmin_media_anual,Tmin_media_mayo_octubre,...,ls_vi6t_max,ls_vi6t_doy_max,ls_vi6t_auc,ls_vi6t_mes_04,ls_vi6t_mes_05,ls_vi6t_mes_06,ls_vi6t_mes_07,ls_vi6t_mes_08,ls_vi6t_mes_09,ls_vi6t_mes_10
0,AGC_001,1360.919123,1494.247836,1337.875435,23.349517,22.812208,22.893978,11.596077,9.978796,11.985211,...,-0.596,157.0,-0.626481,-0.61475,-0.604000,-0.596,-0.657500,-0.624333,NaN,-0.644000
1,AGC_002,961.350809,1053.319641,949.671089,23.780726,23.157021,23.429427,13.328253,11.694942,13.664148,...,-0.580,133.0,-0.622024,-0.61700,-0.607400,-0.588,-0.610667,-0.654250,-0.6260,-0.630500
2,AGC_003,1335.090225,1477.197298,1322.676590,23.801091,23.163777,23.455089,13.277613,11.593201,13.642318,...,-0.612,157.0,-0.641865,-0.65450,-0.616000,-0.612,-0.661000,-0.653000,-0.6450,-0.643667
3,AGC_004,887.946089,966.084870,875.289787,23.791028,23.134377,23.400179,12.122947,10.367646,12.559611,...,-0.610,261.0,-0.627673,-0.62500,-0.636500,-0.616,-0.625000,-0.630000,-0.6230,-0.640000
4,AGC_005,793.730323,868.425233,784.052917,23.513994,22.966181,23.135269,13.248211,11.715880,13.529122,...,-0.611,133.0,-0.637416,-0.64000,-0.628333,-0.617,-0.639333,-0.651250,-0.6405,-0.640500


In [75]:
print(df_modelo.columns)

Index(['ID_POLIGON', 'precipitacion_acumulada_abril_octubre',
       'precipitacion_acumulada_anual', 'precipitacion_acumulada_mayo_octubre',
       'Tmax_media_abril_octubre', 'Tmax_media_anual',
       'Tmax_media_mayo_octubre', 'Tmin_media_abril_octubre',
       'Tmin_media_anual', 'Tmin_media_mayo_octubre',
       ...
       'ls_vi6t_max', 'ls_vi6t_doy_max', 'ls_vi6t_auc', 'ls_vi6t_mes_04',
       'ls_vi6t_mes_05', 'ls_vi6t_mes_06', 'ls_vi6t_mes_07', 'ls_vi6t_mes_08',
       'ls_vi6t_mes_09', 'ls_vi6t_mes_10'],
      dtype='str', length=103)


In [76]:
# Revisión rápida SOLO con entrenamiento: ¿qué variables se relacionan con el rendimiento?
train = df_modelo[df_modelo["CONJUNTO"] == "ENTRENAMIENTO"]
num = train.select_dtypes("number").drop(columns=["rendimiento_t_ha"])
corr = num.corrwith(train["rendimiento_t_ha"], method="spearman").dropna()
print(corr.reindex(corr.abs().sort_values(ascending=False).index).head(15).round(3))

Tmin_media_anual                         0.675
amplitud_termica_media_anual            -0.674
amplitud_termica_media_abril_octubre    -0.669
amplitud_termica_media_mayo_octubre     -0.660
ls_vi6t_n_validos                        0.584
s2_ndvi_mes_05                           0.581
Tmin_media_abril_octubre                 0.542
s2_ndvi_mes_06                           0.509
Tmin_media_mayo_octubre                  0.504
pl_ndvi_mes_05                           0.447
precipitacion_acumulada_abril_octubre   -0.394
precipitacion_acumulada_mayo_octubre    -0.394
precipitacion_acumulada_anual           -0.385
s2_ndvi_mes_04                           0.375
ls_vi6t_mes_07                           0.372
dtype: float64


In [77]:
df_modelo.to_csv("dataset_modelo_cebada.csv", index=False)
df_modelo[df_modelo["CONJUNTO"] == "ENTRENAMIENTO"].to_csv("dataset_entrenamiento.csv", index=False)
df_modelo[df_modelo["CONJUNTO"] == "PREDICCION"].drop(columns="rendimiento_t_ha").to_csv("dataset_prediccion.csv", index=False)